# 400 YouDo 2 — Berat Altun

In [1]:
import numpy as np
import pandas as pd

In [2]:
!head ratings_long.csv

userId,movieId,rating
0,16,5
0,72,5
0,86,5
0,259,1
0,319,4
0,521,4
0,534,2
0,671,1
0,673,2


In [3]:
r = np.full((20, 1000),fill_value=np.nan)

In [4]:
df = pd.read_csv('ratings_long.csv')

In [5]:
for rec in df.itertuples():
    r[rec.userId][rec.movieId] = rec.rating

Note that $r$ matrix is $20 \times 1000$ with only <1\% full (highly sparse)

In [6]:
r

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan,  4., nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])

What if we define two matricies
- u = $20 \times 4$
- v = $4 \times 1000$


Then model $r$ as $u \times v$

Problem is we have to learn for $20 \times 4 + 4 \times 1000 = 4080$ parameters (better than 20.000 x 0.99 missing values)

## Problem

1. Define a convex loss function wrt $u$ and $v$
- Solve using gradient descent algorithm explained in **I Do**
- Use any regulatizer $L1$ or $L2$ to prevent overfitting

## Çözüm

### 1. Deney tasarımı: gözlemlenen puanları eğitim/test ayırmak

Matris çok seyrek; modeli dürüst değerlendirmek için sadece dolu hücreler alınıp %80 eğitim / %20 test ayrıldı.

Puanlar genel ortalamadan ($\mu$) çıkarılarak merkezleniyor, tahmin $\hat r = \mu + u\cdot v$. Böylece $u\cdot v$ ortalamadan sapmayı öğreniyor; az puanı olan filmlerde işi kolaylaştırıyor.

In [7]:
rng = np.random.RandomState(42)

mask = ~np.isnan(r)                 # dolu hücreler
R = np.nan_to_num(r)               # NaN'ları 0 yap
print(f"Toplam hücre: {r.size}, dolu: {mask.sum()} ({100*mask.sum()/r.size:.2f}%)")

# gözlemlenenlerin %20'si test
obs = np.argwhere(mask)
rng.shuffle(obs)
n_test = int(0.2 * len(obs))

train_mask = np.zeros_like(mask)
test_mask  = np.zeros_like(mask)
for i, j in obs[n_test:]:
    train_mask[i, j] = True
for i, j in obs[:n_test]:
    test_mask[i, j] = True

# eğitim ortalaması ile merkezle
mu = R[train_mask].mean()
print(f"Eğitim: {train_mask.sum()}, Test: {test_mask.sum()}, ortalama puan mu={mu:.3f}")

Toplam hücre: 20000, dolu: 200 (1.00%)
Eğitim: 160, Test: 40, ortalama puan mu=3.150


### 2. Konveks kayıp fonksiyonu ve gradyanlar

Sadece gözlemlenen hücreler üzerinden kare hata + L2 ceza:

$$L(u, v) = \sum_{(i,j)\in \text{gözlem}} \big(r_{ij} - (\mu + u_i \cdot v_j)\big)^2 \;+\; \lambda\big(\lVert u\rVert^2 + \lVert v\rVert^2\big)$$

$u\,v$ ikisi için birden konveks değil ama biri sabitken diğerine göre konveks ; gradient descent iyi çalışıyor.

Hata terimi $E = M \odot (R - \mu - uv)$ olarak tanımlanırsa gradyanlar:

$$\frac{\partial L}{\partial u} = -2\,E\,v^{\top} + 2\lambda u, \qquad \frac{\partial L}{\partial v} = -2\,u^{\top}E + 2\lambda v$$

Güncelleme: `param -= lr * grad`. (L1 için ceza gradyanı $\lambda\,\text{sign}(param)$ olur; fonksiyonda seçilebiliyor.)

In [8]:
def loss_fn(u, v, M):
    err = M * (R - mu - u @ v)
    return (err ** 2).sum()


def train_mf(K=4, lr=0.01, lam=2.0, n_epoch=3000, reg="l2", seed=1, verbose=True):
    rs = np.random.RandomState(seed)
    u = rs.normal(0, 0.1, (r.shape[0], K))   # 20 x K
    v = rs.normal(0, 0.1, (K, r.shape[1]))   # K x 1000

    for ep in range(n_epoch):
        err = train_mask * (R - mu - u @ v)  # sadece eğitim hücrelerinde hata

        g_u = -2 * err @ v.T
        g_v = -2 * u.T @ err

        if reg == "l2":                       # L2: gradyana 2*lam*param eklenir
            g_u += 2 * lam * u
            g_v += 2 * lam * v
        elif reg == "l1":                     # L1: gradyana lam*sign(param) eklenir
            g_u += lam * np.sign(u)
            g_v += lam * np.sign(v)

        u -= lr * g_u
        v -= lr * g_v

        if verbose and ep % 500 == 0:
            print(f"epoch {ep:5d}  eğitim kaybı (loss) = {loss_fn(u, v, train_mask):.2f}")

    return u, v


u, v = train_mf(K=4, lr=0.01, lam=2.0, n_epoch=3000, reg="l2")

epoch     0  eğitim kaybı (loss) = 274.38
epoch   500  eğitim kaybı (loss) = 71.76
epoch  1000  eğitim kaybı (loss) = 71.72
epoch  1500  eğitim kaybı (loss) = 71.71
epoch  2000  eğitim kaybı (loss) = 71.72
epoch  2500  eğitim kaybı (loss) = 71.72


### 3. Regülarizasyon overfitting'i engelliyor mu?

İki model karşılaştırılıyor: cezası çok küçük olan ($\lambda=0.05$) ile düzgün cezalı ($\lambda=2.0$). Ölçüt RMSE. Ayrıca "her şeye ortalamayı tahmin et" baseline'ı ile de kıyaslanıyor.

In [9]:
def rmse(u, v, M):
    err = M * (R - mu - u @ v)
    return np.sqrt((err ** 2).sum() / M.sum())


# az cezalı model
u_of, v_of = train_mf(lam=0.05, n_epoch=3000, verbose=False)
# L2 cezalı model
u_reg, v_reg = train_mf(lam=2.0, n_epoch=3000, verbose=False)

base_test = np.sqrt((test_mask * (R - mu) ** 2).sum() / test_mask.sum())

print("Model                         eğitim RMSE   test RMSE")
print(f"lambda=0.05 (az ceza)         {rmse(u_of, v_of, train_mask):.3f}         {rmse(u_of, v_of, test_mask):.3f}")
print(f"lambda=2.0  (L2 ceza)         {rmse(u_reg, v_reg, train_mask):.3f}         {rmse(u_reg, v_reg, test_mask):.3f}")
print(f"baseline (sadece ortalama)      -            {base_test:.3f}")

Model                         eğitim RMSE   test RMSE
lambda=0.05 (az ceza)         0.017         1.357
lambda=2.0  (L2 ceza)         0.670         1.346
baseline (sadece ortalama)      -            1.289


### Sonuçların yorumu

- λ=0.05: eğitim RMSE 0.017 ama test RMSE 1.357 → klasik overfitting (ezberliyor).
- λ=2.0: eğitim RMSE 0.670, test RMSE 1.346 → ezberleme kırılıyor. Ama test kazancı çok küçük (~%1).
- Baseline (hep ortalama μ=3.15): test RMSE 1.289 → her iki modelden de biraz daha iyi.

Sebep veri çok seyrek . Bu kadar az veriyle gizli faktörler ortalamanın ötesine anlamlı geçemiyor; model baseline'ı geçemiyor. Bu, kurulum hatası değil, sinyalin zayıf olması. Düzenek doğru ; daha çok veriyle baseline'ı geçmesi beklenir.

In [10]:
# 0. kullanıcı için en yüksek 5 tahmin
user_id = 0
pred_row = np.clip(mu + u_reg[user_id] @ v_reg, 1, 5)   # puanlar 1-5 aralığında

top5 = np.argsort(pred_row)[::-1][:5]
print(f"Kullanıcı {user_id} için en yüksek tahmin edilen filmler:")
for m in top5:
    gercek = r[user_id, m]
    print(f"  film {m:4d}: tahmin {pred_row[m]:.2f}"
          + ("" if np.isnan(gercek) else f"  (gerçek puan: {gercek:.0f})"))

Kullanıcı 0 için en yüksek tahmin edilen filmler:
  film   16: tahmin 4.58  (gerçek puan: 5)
  film   86: tahmin 4.23  (gerçek puan: 5)
  film   72: tahmin 4.23  (gerçek puan: 5)
  film  360: tahmin 4.23
  film   22: tahmin 4.18


### Özet

- $r \approx u\cdot v$ modeli, gözlemlenen hücreler üzerinde konveks (biconvex) kare-hata kaybıyla kuruldu.
- Gradyanlar elle türetilip gradient descent ile çözüldü; kaybın düştüğü yazdırıldı.
- L2 regülarizasyonu eğitim/test uçurumunu kapattı ; bu seyreklikte test kazancı küçük. Fonksiyon `reg="l1"` ile L1'i de destekliyor.
- Eğitim/test ayrımı sayesinde modelin baseline'ı henüz geçemediği de görülüyor; asıl kısıt verinin seyrekliği.